# Load Data

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2023-24/gws/merged_gw.csv"
df = pd.read_csv(url)

print(df.head())

In [5]:
print(df.columns)

Index(['name', 'position', 'team', 'xP', 'assists', 'bonus', 'bps',
       'clean_sheets', 'creativity', 'element', 'expected_assists',
       'expected_goal_involvements', 'expected_goals',
       'expected_goals_conceded', 'fixture', 'goals_conceded', 'goals_scored',
       'ict_index', 'influence', 'kickoff_time', 'minutes', 'opponent_team',
       'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards',
       'round', 'saves', 'selected', 'starts', 'team_a_score', 'team_h_score',
       'threat', 'total_points', 'transfers_balance', 'transfers_in',
       'transfers_out', 'value', 'was_home', 'yellow_cards', 'GW'],
      dtype='str')


In [6]:
df = df.sort_values(["name", "GW"])

group = df.groupby("name")

# Lag features (previous week)
df["lag_points"] = group["total_points"].shift(1)
df["lag_minutes"] = group["minutes"].shift(1)

# Rolling averages (last 3 games)
df["avg_points_3"] = group["total_points"].shift(1).rolling(3).mean().reset_index(level=0, drop=True)
df["avg_minutes_3"] = group["minutes"].shift(1).rolling(3).mean().reset_index(level=0, drop=True)
df["avg_goals_3"] = group["goals_scored"].shift(1).rolling(3).mean().reset_index(level=0, drop=True)
df["avg_assists_3"] = group["assists"].shift(1).rolling(3).mean().reset_index(level=0, drop=True)

In [7]:
feature_cols = [
    "value",
    "was_home",
    "avg_points_3",
    "avg_minutes_3",
    "avg_goals_3",
    "avg_assists_3"
]

target_col = "total_points"

In [8]:
model_df = df.dropna(subset=feature_cols + [target_col]).copy()

X = model_df[feature_cols]
y = model_df[target_col]

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", mean_squared_error(y_test, pred) ** 0.5)

MAE: 1.2839350226350617
RMSE: 2.199911740821697


In [12]:
def predict_points(features):
    return model.predict(features.reshape(1, -1))[0]